# Trabalho 1 - Recuperação da Informação

Grupo:

- Arthur Trottmann Ramos (14681052)
- Maicon Chaves Marques (14593530)

## Instalação de Dependências e Carregamento de Dataset

In [1]:
pip install NLTK numpy pandas ir_datasets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import ir_datasets

dataset = ir_datasets.load("cranfield")

## Pré-Processamento

In [3]:
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, RegexpTokenizer
from nltk.stem import PorterStemmer

nltk.download('stopwords')

stemmer = PorterStemmer()


def tokenization(text):
  tokenizer = RegexpTokenizer(r'\w+')
  clean_tokens = tokenizer.tokenize(text)
  return lower_case_normalization(clean_tokens)

def remove_stopwords(words):
  stopwords_set = set(stopwords.words('english'))
  filtered_words = [word for word in words if word not in stopwords_set]
  return filtered_words

def lower_case_normalization(words):
  normalized_words = [word.lower() for word in words]
  return normalized_words

def stemming(words):
  stemmed_words = [stemmer.stem(word) for word in words]
  return stemmed_words

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ArthurRamos\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
def preprocess(text, config_type=0):
  words = tokenization(text)

  if config_type == 1:
    words = remove_stopwords(words)
  if config_type == 2:
    words = stemming(words)
  if config_type == 3:
    words = remove_stopwords(words)
    words = stemming(words)

  return words

## Índice Invertido

In [5]:
class InvertedIndex:
  def __init__(self):
    self.index = {}
    self.document_length = {}
    self.n_documents = 0
    self.avgdl = 0.0

  def build(self, documents, preprocessing_type=0):
    for doc in documents:
      doc_id = doc[0]
      doc_title = doc[1]
      doc_text = doc[2]
      doc_author = doc[3]

      self.n_documents += 1

      title_words = preprocess(doc_title, preprocessing_type)
      text_words = preprocess(doc_text, preprocessing_type)
      author_words = preprocess(doc_author, preprocessing_type)

      self.document_length[doc_id] = len(title_words) + len(text_words) + len(author_words)

      for word in title_words:
        if word not in self.index:
          self.index[word] = {}
        if doc_id not in self.index[word]:
          self.index[word][doc_id] = 0
        self.index[word][doc_id] += 1

    self.avgdl = sum(self.document_length.values()) / len(self.document_length)

## Modelo Probabilístico (BM25)

In [6]:
import math

class BM25:
    def __init__(self, inverted_index, k1=0.5, b=0):
        self.inverted_index = inverted_index
        self.k1 = k1
        self.b = b
    
    def score(self, query, doc_id, preprocessing_type=0):
        score = 0.0
        query_words = preprocess(query, preprocessing_type)
    
        for word in query_words:
            if word in self.inverted_index.index and doc_id in self.inverted_index.index[word]:
                tf = self.inverted_index.index[word][doc_id]
                df = len(self.inverted_index.index[word])
                idf = math.log(1 + ((self.inverted_index.n_documents - df + 0.5) / (df + 0.5)))
                dl = self.inverted_index.document_length[doc_id]
                avgdl = self.inverted_index.avgdl
                score += idf * ((tf * (self.k1 + 1)) / (tf + self.k1 * (1 - self.b + self.b * (dl / avgdl))))
    
        return score

## Modelo Vetorial

## Métricas de Avaliação

In [7]:
def precision_at_k(retrieved_docs, relevant_docs, k):
    retrieved_k = retrieved_docs[:k]
    relevant_retrieved = [doc for doc in retrieved_k if doc in relevant_docs]
    precision = len(relevant_retrieved) / k
    return precision

def recall_at_k(retrieved_docs, relevant_docs, k):
    retrieved_k = retrieved_docs[:k]
    relevant_retrieved = [doc for doc in retrieved_k if doc in relevant_docs]
    recall = len(relevant_retrieved) / len(relevant_docs) if relevant_docs else 0
    return recall

def AP(retrieved_docs, relevant_docs):
    relevant_retrieved = [doc for doc in retrieved_docs if doc in relevant_docs]
    if not relevant_retrieved:
        return 0.0

    precision_sum = 0.0
    for i, doc in enumerate(retrieved_docs):
        if doc in relevant_docs:
            precision_sum += precision_at_k(retrieved_docs, relevant_docs, i + 1)

    average_precision = precision_sum / len(relevant_retrieved)
    return average_precision

## Rodando Modelos

In [8]:
# Para cada query, armazena em sets os documentos relevantes (grau de relevância >= 1) de qrels

relevant_docs_per_query = {}

for qrel in dataset.qrels_iter():
    query_id = qrel[0]
    doc_id = qrel[1]
    relevance_grade = qrel[2]

    if relevance_grade >= 1:
        if query_id not in relevant_docs_per_query:
            relevant_docs_per_query[query_id] = set()
        relevant_docs_per_query[query_id].add(doc_id)

In [9]:
preprocessing_type = 0
k = 10

# Testes com a configuração sem remoção de stopwords e sem stemming
inverted_index = InvertedIndex()
inverted_index.build(dataset.docs_iter(), preprocessing_type=preprocessing_type)

bm25 = BM25(inverted_index, 0.5, 0)
### Instanciação do modelo vetorial ###

metrics_bm25 = {}
metrics_vetorial = {}

for query in dataset.queries_iter():
    query_id = query[0]
    query_text = query[1]

    # Calcula a pontuação BM25 para cada documento
    scores_bm25 = {}

    for doc in dataset.docs_iter():
        doc_id = doc[0]

        score = bm25.score(query_text, doc_id, preprocessing_type=preprocessing_type)
        scores_bm25[doc_id] = score

        ### Cálculo do Score para o modelo vetorial ###

    # Ordena os documentos retornados pelo modelo em ordem decrescente de score
    # retrieved_docs = [doc_idMaiorScore, ..., doc_idMenorScore]
    retrieved_docs_bm25 = sorted(scores_bm25, key=scores_bm25.get, reverse=True)

    metrics_bm25[query_id] = [
        precision_at_k(retrieved_docs_bm25, relevant_docs_per_query.get(query_id, set()), k),
        recall_at_k(retrieved_docs_bm25, relevant_docs_per_query.get(query_id, set()), k),
        AP(retrieved_docs_bm25, relevant_docs_per_query.get(query_id, set()))
    ]

In [13]:
print(metrics_bm25)

{'1': [0.4, 0.14285714285714285, 0.19274359368012212], '2': [0.3, 0.125, 0.14535436774724497], '3': [0.5, 0.625, 0.6358455882352941], '4': [0.1, 0.5, 0.5833333333333334], '5': [0.0, 0.0, 0.027772983272764142], '6': [0.0, 0.0, 0.004740294299081682], '7': [0.3, 0.6, 0.3420853806078862], '8': [0.0, 0.0, 0.058125389220742056], '9': [0.3, 1.0, 1.0], '10': [0.1, 0.125, 0.0710555854805643], '11': [0.3, 0.42857142857142855, 0.1315510584708359], '12': [0.0, 0.0, 0.028633369498956562], '13': [0.0, 0.0, 0.004302638569827616], '14': [0.1, 0.5, 0.3055555555555556], '15': [0.0, 0.0, 0.030315923837889586], '16': [0.1, 0.3333333333333333, 0.341366658155787], '17': [0.1, 0.5, 0.5010626992561105], '18': [0.1, 0.3333333333333333, 0.09844462322338428], '19': [0.1, 0.1111111111111111, 0.02572520744064475], '20': [0.4, 0.4444444444444444, 0.3211810589575518], '21': [0.2, 0.5, 0.30755387243308174], '22': [0.0, 0.0, 0.0011337868480725624], '23': [0.1, 0.03125, 0.12634507997953134], '24': [0.1, 0.3333333333333

In [14]:
# Salva as métricas em um arquivo CSV para utilização em Análises

import os
import pandas as pd

df = pd.DataFrame.from_dict(metrics_bm25, orient='index', columns=['Precision', 'Recall', 'AP'])
df.index.name = 'Query_ID'

df.to_csv('metrics_bm25.csv', index=True)